# 🎸 BOSS IR2 - Parameter Discovery (Active Query)

**Strategia aggiornata:**  
Il BOSS IR-2 NON trasmette automaticamente i cambi. Dobbiamo **interrogarlo attivamente** con comandi Roland SysEx.

## Identity Response ricevuta:
```
7E 10 06 02 41 09 05 00 00 00 00 00 00
         ↑  ↑  ↑
       BOSS  Device Model/Family
```

In [ ]:
import mido
from mido import Message
import time

# Auto-detect
IR2_INPUT = None
IR2_OUTPUT = None

for name in mido.get_input_names():
    if 'BOSS' in name.upper() or 'IR-2' in name.upper():
        IR2_INPUT = name
        break
        
for name in mido.get_output_names():
    if 'BOSS' in name.upper() or 'IR-2' in name.upper():
        IR2_OUTPUT = name
        break

print(f"📥 INPUT:  {IR2_INPUT}")
print(f"📤 OUTPUT: {IR2_OUTPUT}")

In [ ]:
def hex_dump(data):
    return " ".join(f"{b:02X}" for b in data)

def roland_checksum(data):
    """Calcola checksum Roland: 128 - (sum % 128)"""
    return (128 - (sum(data) % 128)) & 0x7F

def send_roland_sysex(outport, inport, command, address, data=None, timeout=1.0):
    """
    Invia comando Roland SysEx e attende risposta.
    
    Formato: F0 41 <dev_id> <model_id> <cmd> <addr_4bytes> [data...] <checksum> F7
    """
    dev_id = 0x10  # Da Identity Response
    model_id = [0x00, 0x00, 0x09, 0x05]  # Dedotto dall'Identity (potrebbe essere diverso)
    
    # Costruisci messaggio
    msg_data = [0x41, dev_id] + model_id + [command]
    
    # Indirizzo (4 bytes)
    if isinstance(address, int):
        addr_bytes = [(address >> 24) & 0x7F, (address >> 16) & 0x7F, 
                      (address >> 8) & 0x7F, address & 0x7F]
    else:
        addr_bytes = list(address)
    
    msg_data.extend(addr_bytes)
    
    # Dati (se presenti)
    if data:
        msg_data.extend(data if isinstance(data, list) else [data])
    
    # Checksum (su address + data)
    checksum_data = addr_bytes + (data if data else [])
    checksum = roland_checksum(checksum_data)
    msg_data.append(checksum)
    
    print(f"📤 TX: {hex_dump(msg_data)}")
    
    # Invia
    outport.send(Message('sysex', data=msg_data))
    
    # Attendi risposta
    start = time.time()
    while time.time() - start < timeout:
        msg = inport.poll()
        if msg and msg.type == 'sysex':
            print(f"📥 RX: {hex_dump(msg.data)}")
            return msg.data
        time.sleep(0.01)
    
    print("⏱️ Timeout - no response")
    return None

print("✅ Funzioni caricate")

## Test 1: Data Request (RQ1)

Proviamo a richiedere dati da vari indirizzi di memoria.

In [ ]:
# Test richiesta dati da indirizzo 0x00 00 00 00
# RQ1 = 0x11 (Data Request)

with mido.open_output(IR2_OUTPUT) as out:
    with mido.open_input(IR2_INPUT) as inp:
        print("Test RQ1 - Address 0x00000000, Size 0x01")
        # RQ1 richiede: address (4 bytes) + size (4 bytes)
        response = send_roland_sysex(out, inp, 0x11, 
                                     address=[0x00, 0x00, 0x00, 0x00],
                                     data=[0x00, 0x00, 0x00, 0x01])  # Size = 1 byte

In [ ]:
# Prova altri indirizzi comuni BOSS
test_addresses = [
    ([0x10, 0x00, 0x00, 0x00], "Temporary Patch"),
    ([0x20, 0x00, 0x00, 0x00], "User Patch 1"),
    ([0x01, 0x00, 0x00, 0x00], "System Settings"),
]

with mido.open_output(IR2_OUTPUT) as out:
    with mido.open_input(IR2_INPUT) as inp:
        for addr, desc in test_addresses:
            print(f"\n{'='*50}")
            print(f"Test: {desc} @ {hex_dump(addr)}")
            response = send_roland_sysex(out, inp, 0x11, 
                                         address=addr,
                                         data=[0x00, 0x00, 0x00, 0x10])  # 16 bytes
            time.sleep(0.5)

## Test 2: Esplorazione Sistematica

Scansioniamo blocchi di indirizzi per trovare dati validi.

In [ ]:
# Scan range 0x00 00 00 00 to 0x00 00 10 00
findings = []

with mido.open_output(IR2_OUTPUT) as out:
    with mido.open_input(IR2_INPUT) as inp:
        for i in range(0, 0x100, 0x10):  # Step 16 bytes
            addr = [0x00, 0x00, (i >> 8) & 0x7F, i & 0x7F]
            
            response = send_roland_sysex(out, inp, 0x11,
                                         address=addr,
                                         data=[0x00, 0x00, 0x00, 0x01])
            
            if response:
                findings.append((hex_dump(addr), response))
                print(f"  ✅ Found data at {hex_dump(addr)}")
            
            time.sleep(0.1)

print(f"\n\n📋 Found {len(findings)} valid addresses")

## Test 3: Monitor Changes

Se troviamo indirizzi validi, leggiamo ripetutamente mentre cambi parametri sul pedale.

In [ ]:
# Monitor address (modifica se hai trovato indirizzi validi)
MONITOR_ADDR = [0x00, 0x00, 0x00, 0x00]
MONITOR_SIZE = 0x20  # 32 bytes

print(f"🎧 Monitoring {hex_dump(MONITOR_ADDR)} for 30 seconds")
print("👉 CAMBIA PARAMETRI SUL PEDALE!")
print("="*50)

last_data = None

with mido.open_output(IR2_OUTPUT) as out:
    with mido.open_input(IR2_INPUT) as inp:
        start = time.time()
        while time.time() - start < 30:
            response = send_roland_sysex(out, inp, 0x11,
                                         address=MONITOR_ADDR,
                                         data=[0x00, 0x00, 0x00, MONITOR_SIZE],
                                         timeout=0.5)
            
            if response and response != last_data:
                print(f"\n🔄 CHANGE DETECTED!")
                print(f"   {hex_dump(response)}")
                last_data = response
            
            time.sleep(0.5)